# 99 — M6 CERTIFIED 永続カタログ

persist 上の **既存** M6 `CERTIFIED` を全件スキャンし、
`campaign_b/_m6_certified_catalog/CATALOG.json` にマージする。

- 再実行で全件再スキャン + 新規追加
- kernel 再起動後もディスク上のカタログが残る
- CERTIFIED を捏造しない（レポートが主張したものだけ）
- Campaign B の `claim_scope` は `SCREENING_ONLY` のままになり得る
- **書き込みあり:** セル 1 で `VALIDATED_RG_PERSIST_ACK` を setdefault し、セル 2 で
  `validate_persistent_root()`（write probe）を必ず呼ぶ（98 の read-only とは異なる）。
- disk quota / Errno 122 で probe やカタログ書き込みが失敗したら、先に strip/reclaim
  （`docs/campaign_b_m3_storage_reclaim.md`）。その場合は既存 `CATALOG.json` の
  読み取り専用表示にフォールバックする。


In [ ]:
from pathlib import Path
import os, sys, json

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'campaign_b' / 'm6_certified_catalog.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/campaign_b/m6_certified_catalog.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
# Required before validate_persistent_root() in cell 2 (99 writes the catalog).
os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)


In [ ]:
from src.runtime import PersistentRootError, validate_persistent_root
from src.campaign_b.m6_certified_catalog import (
    catalog_path,
    load_catalog,
    scan_and_update_catalog,
)

# ACK is set in cell 1. Write probe is required because 99 persists the catalog
# (unlike notebook 98, which is read-only and skips validate_persistent_root).
WRITE_OK = False
RESULT: dict = {}
try:
    validate_persistent_root()
    RESULT = scan_and_update_catalog(PERSIST_ROOT)
    WRITE_OK = True
except (PersistentRootError, OSError) as exc:
    errno = getattr(exc, 'errno', None)
    quota = errno == 122 or 'quota' in str(exc).lower() or 'Disk quota' in str(exc)
    print('ERROR: persistent root validate/write failed:', type(exc).__name__, exc)
    if quota:
        print(
            'Disk quota exceeded (Errno 122). Strip/reclaim M3 checkpoints first '
            '(docs/campaign_b_m3_storage_reclaim.md / scripts), then re-run this '
            'notebook to merge and write CATALOG.json.'
        )
    else:
        print(
            'Fix VALIDATED_RG_PERSIST_ROOT / ACK, free space, then re-run. '
            'Falling back to read-only view of any existing catalog.'
        )
    catalog = load_catalog(PERSIST_ROOT)
    RESULT = {
        'all_certified': catalog.get('entries') or [],
        'newly_found': [],
        'total': catalog.get('total', len(catalog.get('entries') or [])),
        'scanned_at': catalog.get('updated_at'),
        'certification_status': catalog.get('certification_status'),
        'claim_scope': catalog.get('claim_scope'),
        'read_only_fallback': True,
        'write_error': str(exc),
    }

print('catalog', catalog_path(PERSIST_ROOT))
print('write_ok', WRITE_OK)
print(json.dumps({
    'scanned_at': RESULT.get('scanned_at'),
    'total': RESULT.get('total'),
    'newly_found_count': len(RESULT.get('newly_found') or []),
    'certification_status': RESULT.get('certification_status'),
    'claim_scope': RESULT.get('claim_scope'),
    'read_only_fallback': RESULT.get('read_only_fallback', False),
}, indent=2))


In [ ]:
print('=== newly found ===')
print(json.dumps(RESULT.get('newly_found') or [], indent=2, ensure_ascii=False, default=str))
print('=== all certified ===')
print(json.dumps(RESULT.get('all_certified') or [], indent=2, ensure_ascii=False, default=str))
